In [3]:
import json
import csv
import os
import time
import joblib

def build_thinned_dataset():
    print("=== STAN 專案：Gowalla 資料庫嚴格瘦身與重建 ===")
    start_time = time.time()

    # 設定精確的輸入與輸出路徑
    dir_raw = 'Gowalla_dataset'
    dir_out = 'Gowalla_thinned'
    
    # 建立新的獨立資料夾
    os.makedirs(dir_out, exist_ok=True)
    print(f"已確認/建立輸出資料夾: {dir_out}")

    # ==========================================
    # 步驟 1：掃描 Train 與 Test，確立絕對合法的 item_id
    # ==========================================
    print("\n1. 掃描 Session 資料確立合法邊界...")
    valid_items = set()
    
    for filename in ['train_with_timestamps.json', 'test_with_timestamps.json']:
        filepath = os.path.join(dir_raw, filename)
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
            for sample in data:
                # 收集歷史軌跡中的 item_id
                for event in sample['history_events']:
                    valid_items.add(event['item_id'])
                # 收集預測目標的 item_id
                valid_items.add(sample['target_event']['item_id'])
                
    print(f"   -> 在 Train/Test 中共發現 {len(valid_items)} 個獨一無二的 item_id。")

    # ==========================================
    # 步驟 2：瘦身 POI Metadata 並建立原始 ID 白名單
    # ==========================================
    print("\n2. 瘦身 poi_metadata.json...")
    valid_raw_locs = set()
    thinned_metadata = {}
    
    with open(os.path.join(dir_raw, 'poi_metadata.json'), 'r', encoding='utf-8') as f:
        poi_meta = json.load(f)
        for k, v in poi_meta.items():
            if 'item_id' in v and v['item_id'] in valid_items:
                thinned_metadata[k] = v
                valid_raw_locs.add(int(v['raw_poi_id']))
                
    # 儲存瘦身後的 Metadata
    out_meta_path = os.path.join(dir_out, 'thinned_poi_metadata.json')
    with open(out_meta_path, 'w', encoding='utf-8') as f:
        json.dump(thinned_metadata, f, indent=4)
    print(f"   -> 瘦身後的 Metadata 已儲存，包含 {len(thinned_metadata)} 個地點。")

    # ==========================================
    # 步驟 3：過濾原始 txt，產出 CSV 與 PKL 字典
    # ==========================================
    print("\n3. 串流過濾 loc-gowalla_totalCheckins.txt...")
    input_txt = os.path.join(dir_raw, 'loc-gowalla_totalCheckins.txt')
    out_csv_path = os.path.join(dir_out, 'thinned_checkins.csv')
    out_pkl_path = os.path.join(dir_out, 'user_lookup.pkl')
    
    original_count = 0
    thinned_count = 0
    user_lookup_dict = {}

    with open(input_txt, 'r', encoding='utf-8') as fin, \
         open(out_csv_path, 'w', newline='', encoding='utf-8') as fout:
        
        writer = csv.writer(fout)
        writer.writerow(['user_id', 'timestamp', 'latitude', 'longitude', 'raw_location_id'])

        for line in fin:
            original_count += 1
            parts = line.strip().split('\t')
            
            # 確保該行有足夠的欄位
            if len(parts) >= 5:
                raw_loc = int(parts[4])
                
                # 如果該地點在我們的白名單內，則保留
                if raw_loc in valid_raw_locs:
                    uid = int(parts[0])
                    timestamp_str = parts[1]
                    
                    # 寫入 CSV 以供人工檢閱與 Pandas 讀取
                    writer.writerow([uid, timestamp_str, parts[2], parts[3], raw_loc])
                    
                    # 建立反查字典 (以 Tuple 為 Key)
                    user_lookup_dict[(raw_loc, timestamp_str)] = uid
                    thinned_count += 1
            
            # 每掃描兩百萬筆印出一次進度
            if original_count % 2000000 == 0:
                print(f"   已掃描 {original_count:,} 筆原始資料...")

    # 封裝字典為二進位檔
    joblib.dump(user_lookup_dict, out_pkl_path)
    
    # ==========================================
    # 輸出最終報告
    # ==========================================
    end_time = time.time()
    reduction_ratio = 100 - (thinned_count / original_count * 100) if original_count > 0 else 0
    
    print("\n=== 重建完成報告 ===")
    print(f"總耗時: {end_time - start_time:.2f} 秒")
    print(f"原始打卡總數: {original_count:,} 筆")
    print(f"瘦身後打卡數: {thinned_count:,} 筆")
    print(f"空間節省率: {reduction_ratio:.2f}% 的冗餘資料已清除。")
    print(f"\n所有瘦身檔案已儲存至: [{dir_out}] 資料夾中。")

if __name__ == '__main__':
    build_thinned_dataset()

=== STAN 專案：Gowalla 資料庫嚴格瘦身與重建 ===
已確認/建立輸出資料夾: Gowalla_thinned

1. 掃描 Session 資料確立合法邊界...
   -> 在 Train/Test 中共發現 29511 個獨一無二的 item_id。

2. 瘦身 poi_metadata.json...
   -> 瘦身後的 Metadata 已儲存，包含 29511 個地點。

3. 串流過濾 loc-gowalla_totalCheckins.txt...
   已掃描 2,000,000 筆原始資料...
   已掃描 4,000,000 筆原始資料...
   已掃描 6,000,000 筆原始資料...

=== 重建完成報告 ===
總耗時: 128.77 秒
原始打卡總數: 6,442,892 筆
瘦身後打卡數: 1,942,344 筆
空間節省率: 69.85% 的冗餘資料已清除。

所有瘦身檔案已儲存至: [Gowalla_thinned] 資料夾中。
